Se quiere analizar:
•	criptos más negociadas
•	volumen de operaciones 
•	monedas con mayor crecimiento 
•	usuarios con mayores inversiones 
•	riesgos del mercado

a)	Criptos con pérdidas
b)	Inversiones más altas
c)	Agrupar por criptomoneda
d)	Ranking de usuarios
e)	Cripto más rentable
f)	Clasificar Riesgo


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder \
    .appName("Biblioteca") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/21 00:08:01 WARN Utils: Your hostname, DarkosRace, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/21 00:08:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 00:08:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/21 00:08:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
df = spark.read.csv("cripto.csv", header=True, inferSchema=True)
df.show()
# df.printSchema() #? Imprimir estructura de columnas

+--------------+-----------------+------------+--------+-------------+-------------+-----------+----------+
|id_transaccion|          usuario|criptomoneda|cantidad|precio_compra|precio_actual|volumen_24h|     fecha|
+--------------+-----------------+------------+--------+-------------+-------------+-----------+----------+
|          T001|         Ana_Data|         BTC|  0.6198|     60292.84|      62000.0|35000000000|2026-05-12|
|          T002|         Ana_Data|        DOGE| 14088.8|         0.17|         0.14| 2100000000|2026-05-19|
|          T003|      InversorPro|         BTC|  0.5544|     47054.23|      62000.0|35000000000|2026-05-18|
|          T004|        EcoCrypto|         BTC|  1.7929|     56939.16|      62000.0|35000000000|2026-05-18|
|          T005|      InversorPro|         ETH|     3.7|      3276.08|       2900.0|18000000000|2026-05-10|
|          T006|       DevMendoza|        DOGE|  7668.9|         0.16|         0.14| 2100000000|2026-05-13|
|          T007|Lucia_Invers

In [5]:
# df.select("usuario", "criptomoneda", "precio_compra").show()  

#! criptos más negociadas
df.groupBy("criptomoneda") \
.agg(F.count("id_transaccion").alias("transacciones")) \
.orderBy(F.desc("transacciones")) \
.limit(3) \
.show()  


+------------+-------------+
|criptomoneda|transacciones|
+------------+-------------+
|         ETH|            7|
|         SOL|            7|
|         DOT|            6|
+------------+-------------+



In [6]:
#! Volumen de operaciones por fecha y cripto
# Agrupamos por fecha y moneda, y sumamos el volumen
df.groupBy("fecha", "criptomoneda") \
  .agg(F.sum("volumen_24h").alias("volumen_total_dia")) \
  .orderBy("fecha") \
  .show()

+----------+------------+-----------------+
|     fecha|criptomoneda|volumen_total_dia|
+----------+------------+-----------------+
|2026-05-10|         ETH|      36000000000|
|2026-05-10|         SOL|       4500000000|
|2026-05-10|         DOT|        850000000|
|2026-05-11|         BTC|      35000000000|
|2026-05-11|         SOL|       4500000000|
|2026-05-11|         DOT|       1700000000|
|2026-05-11|         ETH|      18000000000|
|2026-05-12|         SOL|       4500000000|
|2026-05-12|         ETH|      18000000000|
|2026-05-12|         ADA|       1200000000|
|2026-05-12|         BTC|      35000000000|
|2026-05-13|         DOT|       1700000000|
|2026-05-13|         ADA|       1200000000|
|2026-05-13|        DOGE|       2100000000|
|2026-05-14|         SOL|       4500000000|
|2026-05-14|         ETH|      36000000000|
|2026-05-15|         BTC|      35000000000|
|2026-05-15|         SOL|       4500000000|
|2026-05-15|        DOGE|       2100000000|
|2026-05-18|         SOL|       

In [7]:
#! Usuarios con mayores inversiones (3 usuarios con mas)

# df.select("usuario", "criptomoneda", "cantidad", "precio_compra") \
#     .orderBy(F.desc("cantidad")) \
#     .limit(4) \
#     .show()

#? ↗ Ese codigo cuenta las mayores inversiones, pero se pueden repetir los usuarios,
#? ↘ Este cuenta los usuarios con mayores inversiones sumando todas las de cada usuario individualmente y los agrupa

#* 1. Creamos una columna temporal con el dinero real invertido en cada fila
df_con_inversion = df.withColumn("dinero_invertido", F.col("cantidad") * F.col("precio_compra"))
#* 2. Agrupamos por usuario (así no se repiten!) y sumamos todo su dinero
df_con_inversion.groupBy("usuario") \
    .agg(F.sum("dinero_invertido").alias("total_invertido")) \
    .orderBy(F.desc("total_invertido")) \
    .limit(3) \
    .show()

#? Riesgos del mercado (?)

+-----------+------------------+
|    usuario|   total_invertido|
+-----------+------------------+
|  EcoCrypto|137665.33696400002|
|InversorPro| 92019.47451200002|
|  Matias_99| 83798.64126499998|
+-----------+------------------+



In [8]:
#! Criptos con perdidas o ganancias
dfConPerdidas = df.withColumn("Diferencia", F.col("precio_actual") - F.col("precio_compra"))

# dfConPerdidas.filter(F.col("Diferencia") < 0).show()

dfConPerdidas.groupBy("criptomoneda") \
    .agg(
        F.sum(F.when(F.col("Diferencia") < 0, 1).otherwise(0)).alias("Negativas"),
        F.sum(F.when(F.col("Diferencia") > 0, 1).otherwise(0)).alias("Positivas")
    ) \
    .show()

+------------+---------+---------+
|criptomoneda|Negativas|Positivas|
+------------+---------+---------+
|         DOT|        6|        0|
|         ETH|        6|        1|
|        DOGE|        4|        0|
|         BTC|        0|        5|
|         SOL|        1|        6|
|         ADA|        4|        1|
+------------+---------+---------+



In [9]:
#! Monedas con mayor crecimiento / Cipto más rentable
# Calculamos el porcentaje de ganancia/pérdida sobre el precio de compra
df_rentabilidad = dfConPerdidas.withColumn(
    "Rendimiento_Porcentual", 
    ((F.col("precio_actual") - F.col("precio_compra")) / F.col("precio_compra")) * 100
)

# Agrupamos por criptomoneda para sacar el promedio de crecimiento
df_rentabilidad.groupBy("criptomoneda") \
    .agg(F.round(F.avg("Rendimiento_Porcentual"), 2).alias("Crecimiento_Promedio_%")) \
    .orderBy(F.desc("Crecimiento_Promedio_%")) \
    .show()

+------------+----------------------+
|criptomoneda|Crecimiento_Promedio_%|
+------------+----------------------+
|         BTC|                 14.58|
|         SOL|                  7.74|
|         ETH|                 -10.6|
|        DOGE|                -11.03|
|         ADA|                -13.04|
|         DOT|                -20.58|
+------------+----------------------+



In [10]:
#! Riesgo del mercado

# Creamos la columna riesgo basada en el volumen global de transacciones
dfRiesgo = df.withColumn(
    "Clasificacion_Riesgo",
    F.when(F.col("volumen_24h") > 10000000000, "Bajo")
     .when(F.col("volumen_24h") >= 2000000000, "Medio")
     .otherwise("Alto")
)

# Mostramos un resumen limpio de cada moneda con su nivel de riesgo
dfRiesgo.select("criptomoneda", "volumen_24h", "Clasificacion_Riesgo").distinct().show()

+------------+-----------+--------------------+
|criptomoneda|volumen_24h|Clasificacion_Riesgo|
+------------+-----------+--------------------+
|         SOL| 4500000000|               Medio|
|         DOT|  850000000|                Alto|
|         ETH|18000000000|                Bajo|
|        DOGE| 2100000000|               Medio|
|         BTC|35000000000|                Bajo|
|         ADA| 1200000000|                Alto|
+------------+-----------+--------------------+

